# GRPC-based Hyperspectral Image Classification (Indian Pines)

Based on : [A new hyperspectral image classification method based on spatial-spectral features](https://rdcu.be/eYsGq)

In [28]:

import numpy as np
import pandas as pd
import matplotlib as plt
from scipy.io import loadmat
from scipy import signal
from scipy.ndimage import convolve
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, cohen_kappa_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [29]:
# ========== 1. LOAD DATA ==========
DATA_PATH = "../dataset/Indian_pines_corrected (1).mat"
GT_PATH = "../dataset/Indian_pines_gt.mat"

data = loadmat(DATA_PATH)
cube = data['indian_pines_corrected']

gt = loadmat(GT_PATH)
gt_map = gt['indian_pines_gt']

print(f"HSI Cube shape: {cube.shape}")
print(f"Ground Truth shape: {gt_map.shape}")

H, W, B = cube.shape
X_all = cube.reshape(H * W, B)
y_all = gt_map.reshape(H * W)

labeled_indices = np.where(y_all != 0)[0]
X = X_all[labeled_indices, :]
y = y_all[labeled_indices]

band_columns = [f"band_{i+1}" for i in range(B)]
df = pd.DataFrame(X, columns=band_columns)
df["class"] = y

print(f"\nTotal labeled samples: {len(y)}")
print(f"Number of spectral bands: {X.shape[1]}")
print(f"Number of classes: {len(np.unique(y))}")

HSI Cube shape: (145, 145, 200)
Ground Truth shape: (145, 145)

Total labeled samples: 10249
Number of spectral bands: 200
Number of classes: 16


In [30]:
df.head()

,band_1,band_2,band_3,band_4,band_5,band_6,band_7,band_8,band_9,band_10,...,band_192,band_193,band_194,band_195,band_196,band_197,band_198,band_199,band_200,class
0,3172,4142,4506,4279,4782,5048,5213,5106,5053,4750,...,1094,1090,1112,1090,1062,1069,1057,1020,1020,3
1,2580,4266,4502,4426,4853,5249,5352,5353,5347,5065,...,1108,1104,1117,1091,1079,1085,1064,1029,1020,3
2,3687,4266,4421,4498,5019,5293,5438,5427,5383,5132,...,1111,1114,1114,1100,1065,1092,1061,1030,1016,3
3,2749,4258,4603,4493,4958,5234,5417,5355,5349,5096,...,1122,1108,1109,1109,1071,1088,1060,1030,1006,3
4,2746,4018,4675,4417,4886,5117,5215,5096,5098,4834,...,1110,1107,1112,1094,1072,1087,1052,1034,1019,3


## Dimensiality Reduction

In [3]:
n_pca = 3
pca = PCA(n_components=n_pca)
X_all_pca = pca.fit_transform(X_all)
X_pca = X_all_pca[labeled_indices, :]
print(f"PCA: {B} -> {n_pca} components")

PCA: 200 -> 3 components


In [4]:
# LDA to reduce to number of classes - 1
n_lda = min(len(np.unique(y)) - 1, n_pca)
lda = LDA(n_components=n_lda)
X_lda = lda.fit_transform(X_pca, y)
X_all_lda = lda.transform(X_all_pca)
print(f"LDA: {n_pca} -> {n_lda} components")


reduced_cube = X_all_lda.reshape(H, W, n_lda)

LDA: 3 -> 3 components


## Gabor Filter

In [5]:
def gabor_kernel(frequency, theta, sigma_x=1, sigma_y=1, n_stds=3):
    """
    Generate a 2D Gabor kernel.
    
    Parameters:
    -----------
    frequency : float
        Spatial frequency of the harmonic function
    theta : float
        Orientation of the Gabor function in radians
    sigma_x : float
        Standard deviation in x-direction
    sigma_y : float
        Standard deviation in y-direction
    n_stds : int
        Number of standard deviations to include in the kernel size
    
    Returns:
    --------
    kernel : ndarray
        2D Gabor kernel
    """
    # Calculate kernel size based on standard deviations
    size = int(2 * n_stds * max(sigma_x, sigma_y) + 1)
    if size % 2 == 0:
        size += 1
    
    # Create coordinate grids
    x, y = np.meshgrid(np.arange(size) - size // 2, 
                       np.arange(size) - size // 2)
    
    # Rotate coordinates
    x_theta = x * np.cos(theta) + y * np.sin(theta)
    y_theta = -x * np.sin(theta) + y * np.cos(theta)
    
    # Create Gabor kernel
    g = np.exp(-0.5 * (x_theta**2 / sigma_x**2 + y_theta**2 / sigma_y**2))
    g *= np.cos(2 * np.pi * frequency * x_theta)
    
    return g

def apply_gabor_bank(img, frequencies=[0.2], orientations=4, sigma_x=1, sigma_y=1):
    """
    Apply Gabor filter bank to extract spatial features.
    Based on the paper methodology.
    """
    gabor_features = []
    
    # Ensure image is float
    img = img.astype(np.float64)
    
    # Apply smoothing first (as shown in user's selection)
    img = convolve(img, np.ones((3, 3)) / 9, mode='reflect')
    
    for freq in frequencies:
        for theta_idx in range(orientations):
            theta = theta_idx * np.pi / orientations
            # Create Gabor kernel
            kernel = np.real(gabor_kernel(
                frequency=freq,
                theta=theta,
                sigma_x=sigma_x,
                sigma_y=sigma_y
            ))
            # Apply filter with 'valid' mode as per paper
            filtered = signal.convolve2d(img, kernel, mode='valid')
            gabor_features.append(filtered)
    
    return np.stack(gabor_features, axis=-1)

In [6]:
# Apply Gabor filters to each LDA component
# Based on the paper: apply Gabor filters with 4 orientations [0, π/4, π/2, 3π/4]
thetas = [0, np.pi/4, np.pi/2, 3*np.pi/4]
gabor_maps = []

for d in range(n_lda):
    band = reduced_cube[:, :, d]
    # Apply smoothing (as per user's selection)
    band = convolve(band, np.ones((3, 3)) / 9, mode='reflect')
    
    for theta in thetas:
        kernel = np.real(gabor_kernel(
            frequency=0.2,
            theta=theta,
            sigma_x=1,
            sigma_y=1
        ))
        
        filtered = signal.convolve2d(band, kernel, mode='valid')
        gabor_maps.append(filtered)

# Stack all Gabor features
gabor_cube = np.stack(gabor_maps, axis=-1)
print(f"Gabor features shape: {gabor_cube.shape}")

# Update H and W to match the 'valid' convolution output
H_gabor, W_gabor = gabor_cube.shape[:2]
print(f"Updated spatial dimensions after Gabor: ({H_gabor}, {W_gabor})")

Gabor features shape: (139, 139, 12)
Updated spatial dimensions after Gabor: (139, 139)


In [7]:
def pca_whitening(X, n_components=None):
    """PCA whitening for data normalization"""
    if n_components is None:
        n_components = min(X.shape)
    
    pca = PCA(n_components=n_components, whiten=True)
    X_white = pca.fit_transform(X)
    return X_white, pca

## Random Patch Convolution

In [8]:
def random_patch_layer(data_cube, n_patches=23, patch_size=24, random_state=None):
    """
    Single layer of random patch convolution.
    Based on the paper methodology.
    """
    if random_state is not None:
        np.random.seed(random_state)
    
    H, W, C = data_cube.shape
    
    # Flatten spatial dimensions for PCA whitening
    X = data_cube.reshape(H * W, C)
    
    # PCA whitening
    n_comp = min(C, H * W - 1)
    X_white, _ = pca_whitening(X, n_components=n_comp)
    data_white = X_white.reshape(H, W, n_comp)
    
    # Random patch extraction and convolution
    feature_maps = []
    
    for i in range(n_patches):
        # Randomly select a pixel location
        h_rand = np.random.randint(0, max(1, H - patch_size + 1))
        w_rand = np.random.randint(0, max(1, W - patch_size + 1))
        
        # Extract patch (handle edge cases)
        h_end = min(h_rand + patch_size, H)
        w_end = min(w_rand + patch_size, W)
        patch = data_white[h_rand:h_end, w_rand:w_end, :]
        
        # Ensure patch has the correct size (pad if necessary)
        if patch.shape[0] < patch_size or patch.shape[1] < patch_size:
            pad_h = patch_size - patch.shape[0]
            pad_w = patch_size - patch.shape[1]
            patch = np.pad(patch, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')
            patch = patch[:patch_size, :patch_size, :]
        
        # Convolve each channel with the patch
        conv_result = np.zeros((H, W))
        for c in range(n_comp):
            patch_2d = patch[:, :, c]
            data_2d = data_white[:, :, c]
            # Use 'same' mode to maintain spatial dimensions
            conv = signal.convolve2d(data_2d, patch_2d, mode='same', boundary='symm')
            conv_result += conv
        
        feature_maps.append(conv_result)
    
    feature_cube = np.stack(feature_maps, axis=-1)
    
    # Modified ReLU activation: f(I) = max(0, I - mean(I))
    mean_vals = np.mean(feature_cube, axis=(0, 1), keepdims=True)
    feature_cube = np.maximum(0, feature_cube - mean_vals)
    
    return feature_cube

In [9]:
# Multi-layer random patch convolution (6 layers as per paper)
n_layers = 6
rpc_layers = []

# Set random seed for reproducibility
np.random.seed(42)

current_input = gabor_cube
for layer in range(n_layers):
    print(f"  Layer {layer + 1}/{n_layers}...")
    layer_features = random_patch_layer(current_input, n_patches=23, patch_size=24, random_state=42+layer)
    rpc_layers.append(layer_features)
    current_input = layer_features
    print(f"    Output shape: {layer_features.shape}")

print(f"\nRandom Patch Convolution completed with {n_layers} layers")

  Layer 1/6...
    Output shape: (139, 139, 23)
  Layer 2/6...
    Output shape: (139, 139, 23)
  Layer 3/6...
    Output shape: (139, 139, 23)
  Layer 4/6...
    Output shape: (139, 139, 23)
  Layer 5/6...
    Output shape: (139, 139, 23)
  Layer 6/6...
    Output shape: (139, 139, 23)

Random Patch Convolution completed with 6 layers


## Feature Fusion

In [10]:
# Flatten Gabor features
H_gabor, W_gabor, C_gabor = gabor_cube.shape
gabor_flat = gabor_cube.reshape(H_gabor * W_gabor, -1)

# Flatten RPC layer features
rpc_features_list = []
for layer_feat in rpc_layers:
    H_rpc, W_rpc, C_rpc = layer_feat.shape
    rpc_features_list.append(layer_feat.reshape(H_rpc * W_rpc, -1))

# Handle dimension mismatch: need to align spatial dimensions
# The Gabor output has different dimensions due to 'valid' convolution
# We need to map the original labeled indices to the new dimensions

# Calculate the offset from 'valid' convolution
# Gabor kernel size: 2 * n_stds * max(sigma_x, sigma_y) + 1
# With sigma_x=1, sigma_y=1, n_stds=3: kernel_size = 2*3*1+1 = 7
kernel_size = int(2 * 3 * max(1, 1) + 1)
if kernel_size % 2 == 0:
    kernel_size += 1
offset = kernel_size // 2  # offset = 3 for kernel_size = 7

# Create mapping from original indices to Gabor-filtered indices
def map_indices_to_gabor(original_indices, H_orig, W_orig, H_gabor, W_gabor, offset):
    """Map original pixel indices to Gabor-filtered indices"""
    gabor_indices = []
    valid_mask = []
    
    for idx in original_indices:
        h_orig = idx // W_orig
        w_orig = idx % W_orig
        
        # Check if pixel is within valid convolution region
        if (h_orig >= offset and h_orig < H_orig - offset and 
            w_orig >= offset and w_orig < W_orig - offset):
            h_gabor = h_orig - offset
            w_gabor = w_orig - offset
            gabor_idx = h_gabor * W_gabor + w_gabor
            gabor_indices.append(gabor_idx)
            valid_mask.append(True)
        else:
            gabor_indices.append(-1)  # Invalid index
            valid_mask.append(False)
    
    return np.array(gabor_indices), np.array(valid_mask)

# Map indices
gabor_labeled_indices, valid_mask = map_indices_to_gabor(
    labeled_indices, H, W, H_gabor, W_gabor, offset
)

# Filter to only valid samples
valid_gabor_indices = gabor_labeled_indices[valid_mask]
valid_y = y[valid_mask]

# Stack: Gabor + all RPC layers + original spectral (LDA-reduced)
# Note: All RPC layers should have same spatial dimensions as Gabor output
fused_features_gabor = np.hstack([gabor_flat] + rpc_features_list)

# Add LDA features - need to map them to Gabor dimensions too
X_all_lda_gabor = X_all_lda.reshape(H, W, n_lda)
X_all_lda_gabor_valid = X_all_lda_gabor[offset:H-offset, offset:W-offset, :]
X_all_lda_gabor_flat = X_all_lda_gabor_valid.reshape(H_gabor * W_gabor, -1)

# Final feature fusion
fused_features = np.hstack([fused_features_gabor, X_all_lda_gabor_flat])

# Standardize features
scaler = StandardScaler()
fused_features = scaler.fit_transform(fused_features)

# Extract labeled samples (only valid ones)
X_final = fused_features[valid_gabor_indices, :]
y_final = valid_y

print(f"Valid samples after Gabor filtering: {len(y_final)}/{len(y)}")
print(f"Final feature shape: {X_final.shape}")

Valid samples after Gabor filtering: 9994/10249
Final feature shape: (9994, 153)


In [18]:
indices = np.arange(len(y_final))

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_final,
    y_final,
    indices,
    test_size=0.7,
    random_state=42,
    stratify=y_final
)

print(f"Training samples: {len(y_train)}")
print(f"Testing samples: {len(y_test)}")

Training samples: 2998
Testing samples: 6996


In [19]:
# Train SVM with RBF kernel
svm = SVC(kernel='rbf', C=1000, gamma='scale', random_state=42)
svm.fit(X_train, y_train)

# Predict
y_pred = svm.predict(X_test)

In [20]:
# ========== 7. EVALUATION ==========
print("\n=== Results ===")
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"Kappa Coefficient: {cohen_kappa_score(y_test, y_pred):.4f}")

print("\nPer-class accuracy:")
print(classification_report(y_test, y_pred, target_names=[
    'Alfalfa', 'Corn-notill', 'Corn-mintill', 'Corn', 'Grass-pasture',
    'Grass-trees', 'Grass-pasture-mowed', 'Hay-windrowed', 'Oats',
    'Soybean-notill', 'Soybean-mintill', 'Soybean-clean', 'Wheat',
    'Woods', 'Buildings-Grass-Trees-Drives', 'Stone-Steel-Towers'
]))


=== Results ===
Overall Accuracy: 99.91%
Kappa Coefficient: 0.9990

Per-class accuracy:
                              precision    recall  f1-score   support

                     Alfalfa       1.00      1.00      1.00        32
                 Corn-notill       1.00      1.00      1.00      1000
                Corn-mintill       1.00      1.00      1.00       527
                        Corn       1.00      1.00      1.00       166
               Grass-pasture       1.00      1.00      1.00       317
                 Grass-trees       1.00      1.00      1.00       511
         Grass-pasture-mowed       1.00      0.95      0.97        20
               Hay-windrowed       1.00      1.00      1.00       335
                        Oats       1.00      1.00      1.00        14
              Soybean-notill       1.00      1.00      1.00       673
             Soybean-mintill       1.00      1.00      1.00      1674
               Soybean-clean       1.00      1.00      1.00       413


In [ ]:
def create_and_display_classification_map(
    y_train, y_test, predictions,
    indices_train, indices_test,
    df_filtered, title
):
    """
    Corrected classification map reconstruction
    """

    # 1️⃣ Allocate map over ALL valid pixels
    clmap_filtered = np.zeros(len(df_filtered), dtype=int)

    # 2️⃣ Fill training pixels
    for i, idx in enumerate(indices_train):
        clmap_filtered[idx] = y_train[i]

    # 3️⃣ Fill test pixels with predictions
    for i, idx in enumerate(indices_test):
        clmap_filtered[idx] = predictions[i]

    # 4️⃣ Map back to original image grid
    H, W = 145, 145
    clmap = np.zeros(H * W, dtype=int)

    original_indices = df_filtered.index.values
    for i, orig_idx in enumerate(original_indices):
        clmap[orig_idx] = clmap_filtered[i]

    # 5️⃣ Display
    plt.figure(figsize=(8, 6))
    plt.imshow(clmap.reshape(H, W), cmap='jet')
    plt.colorbar(label='Class')
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    return clmap


In [32]:
create_and_display_classification_map(y_train,y_test,y_pred,idx_train,idx_test,df,"GABOR FIlter method Classification Map")

Mapping training data:   0%|          | 0/2998 [00:00<?, ?it/s]


IndexError: index 4708 is out of bounds for axis 0 with size 2998